# Colab에서 YOLOv8 쓰레기 분류 모델 학습하기

순서대로 셀을 실행하면 데이터셋 다운로드부터 클래스 변환, 모델 훈련 및 `best.pt` 다운로드까지 원스톱으로 진행됩니다. (상단 메뉴의 **런타임 > 런타임 유형 변경 > 하드웨어 가속기를 T4 GPU** 로 설정하고 진행하세요)

In [ ]:
# 1. 필요한 라이브러리 설치 및 데이터셋 다운로드
!pip install ultralytics
!git clone https://huggingface.co/datasets/hyeon2525/Ku-Yolo-DataSet

In [ ]:
# 2. 데이터 라벨 5개 클래스 -> 4개 클래스로 변환 스크립트 실행
import os
from pathlib import Path
from tqdm import tqdm

INPUT_ROOT = 'Ku-Yolo-DataSet/labels_5classes'
OUTPUT_ROOT = 'Ku-Yolo-DataSet/labels_4classes'

NEW_CLASSES = ["can", "plastic", "paper", "bottle"]

CLASS_MAPPING = {
    0: 1,     # Plastic -> plastic
    1: None,  # Vinyl -> 삭제 (추가 안함)
    2: 0,     # Can -> can
    3: 3,     # Glass -> bottle
    4: 2,     # Paper -> paper
}

input_path = Path(INPUT_ROOT)
output_path = Path(OUTPUT_ROOT)

label_files = list(input_path.rglob('*.txt'))
label_files = [f for f in label_files if f.name not in ['classes.txt', 'classes_kr.txt']]

print(f"Found {len(label_files)} label files. Converting...")
converted_count = 0
for file_path in tqdm(label_files, desc="Converting"):
    with open(file_path, 'r') as f:
        lines = f.readlines()
        
    new_lines = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5: continue
        
        old_id = int(parts[0])
        new_id = CLASS_MAPPING.get(old_id, None)
        
        if new_id is not None:
            coords = " ".join(parts[1:])
            new_lines.append(f"{new_id} {coords}")
            
    relative_path = file_path.relative_to(input_path)
    target_path = output_path / relative_path
    target_path.parent.mkdir(parents=True, exist_ok=True)
    
    if new_lines:
        with open(target_path, 'w') as f:
            f.write('\n'.join(new_lines))
    else:
        target_path.touch()
    converted_count += 1

with open(output_path / 'classes.txt', 'w') as f:
    f.write('\n'.join(NEW_CLASSES))

print(f"\n라벨 변환 완료: {converted_count} files processed.")

In [ ]:
# 3. data.yaml 생성 (Colab 기본 경로 /content/ 기준)
yaml_content = """path: /content/Ku-Yolo-DataSet
train: data
val: data
nc: 4
names:
  - can
  - plastic
  - paper
  - bottle
"""
with open('Ku-Yolo-DataSet/labels_4classes/data.yaml', 'w') as f:
    f.write(yaml_content)
    
print("data.yaml 파일이 생성되었습니다.")

In [ ]:
# 4. CUDA(GPU) 환경 확인 및 YOLO 모델 훈련 시작
import torch
from ultralytics import YOLO

print(f"사용 중인 GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "GPU가 비활성화 되어있습니다!")

# Pre-trained 모델 로드
model = YOLO("yolov8n.pt")

# 학습 파라미터 지정 (충분한 성능 향상을 위해 100 에포크 추천, early stopping 이 내장되어 있으므로 켜두고 자도 무방함) 
results = model.train(
    data="/content/Ku-Yolo-DataSet/labels_4classes/data.yaml",
    epochs=100,       # 학습 횟수
    patience=20,      # 성능 개선이 20 에포크 이상 없으면 조기 종료 (Early Stopping)
    imgsz=640,
    batch=16,
    device=0,        # 코랩 GPU 사용
    project="trash_classification_4classes",
    name="yolov8n_colab_train"
)

In [ ]:
# 5. 학습이 끝난 가장 좋은 모델(best.pt) 파일 다운로드
from google.colab import files

best_model_path = "trash_classification_4classes/yolov8n_colab_train/weights/best.pt"
print(f"최고 성능 모델 다운로드를 시작합니다: {best_model_path}")

files.download(best_model_path)